<a href="https://colab.research.google.com/github/almagomen/v2v/blob/monorepo-subtree/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video2Video – Despliegue en Colab

1. **Activa GPU**: *Runtime → Change runtime type → T4 GPU*
2. Ejecuta las celdas en orden.
3. Configura **Secrets** (panel izquierdo 🔑): `GOOGLE_API_KEY`, `LMNT_API_KEY`

## 1. Clonar repo e instalar dependencias

In [1]:
# @title ⚠️ Un solo paso: clonar monorepo + construir OpenReel + levantar API (con túnel)
%cd /content
import os, time, threading, sys, warnings, subprocess
import uvicorn
from pathlib import Path
from google.colab import userdata, drive, output
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)

# CONFIGURACIÓN
if not os.path.exists("/content/drive"): drive.mount("/content/drive")

GHP_TOKEN = userdata.get("GHP_GITHUB_TOKEN")
REPO_URL = f"https://{GHP_TOKEN}@github.com/almagomen/v2v.git"
REPO_PATH = Path("/content/v2v")
BRANCH_NAME = "monorepo-subtree"  # OpenReel subtree
BACKEND_PORT = 8000
FRONTEND_PORT = 5173

# Clonar/actualizar
if not REPO_PATH.exists():
    print("📥 Clonando repositorio...")
    !git clone -q -b {BRANCH_NAME} {REPO_URL} {REPO_PATH}
else:
    print("🔄 Actualizando repositorio...")
    !git -C {REPO_PATH} fetch origin {BRANCH_NAME} -q
    !git -C {REPO_PATH} reset --hard origin/{BRANCH_NAME} -q

if str(REPO_PATH) not in sys.path: sys.path.insert(0, str(REPO_PATH))

# Instalación de Dependencias
if 'deps_installed' not in globals():
    print("📦 Instalando dependencias...")
    !npm install -g pnpm ---silent
    !uv pip install git+https://github.com/jianfch/stable-ts.git git+https://github.com/pyannote/pyannote-audio.git --no-deps -q
    !uv pip install yt-dlp bs-roformer-infer tinydb sentencex torchcodec==0.7.0  'torch-audiomentations>=0.12.0'  'torchmetrics>=1.6.1' 'optuna>=4.2.0' -q
    !uv pip install openai-whisper 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q
    !uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0'  'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1'  -q

    print("🛠️ Cargando modelos...")
    from src.stt import get_whisper_model
    from src.diarization import get_diarization_model
    from src.download import get_roformer_model
    get_whisper_model()
    get_diarization_model()
    get_roformer_model()
    globals()['deps_installed'] = True

# Build OpenReel
OPENREEL_PATH = REPO_PATH / "openreel"
FRONTEND_MODE = "dev"  # "dev" para desarrollo, "preview" para ver el build

print("🏗️  Preparando OpenReel (deps)...")
!cd {OPENREEL_PATH} && pnpm install -silent

if FRONTEND_MODE == "preview":
    print("🏗️  Construyendo OpenReel (WASM + web)...")
    !cd {OPENREEL_PATH} && pnpm build

print("\n🛠️ Iniciando APP...")

# Backend FastAPI
class ControlledServer(uvicorn.Server):
    def install_signal_handlers(self):
        return

def run_backend():
    global server
    from api.main import app
    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT)
    server = ControlledServer(config=config)
    server.run()

if 'server' in globals() and server.started:
    server.should_exit = True
    time.sleep(1)

thread = threading.Thread(target=run_backend, daemon=True)
thread.start()

# Lanzar FRONTEND
!fuser -k {FRONTEND_PORT}/tcp 2>/dev/null

frontend_cmd = (
    [
        "pnpm",
        ("dev" if FRONTEND_MODE == "dev" else "preview"),
        "--",
        "--host",
        "0.0.0.0",
        "--port",
        str(FRONTEND_PORT),
    ]
)

process = subprocess.Popen(
    frontend_cmd,
    cwd=str(OPENREEL_PATH),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

if 'vite_history' not in globals():
    vite_history = []

def log_reader(pipe):
    for line in iter(pipe.readline, ''):
        clean_line = line.strip()
        if clean_line: vite_history.append(clean_line)

threading.Thread(target=log_reader, args=(process.stdout,), daemon=True).start()

# Túnel Nativo
print("\n✨ ENTORNO LISTO ✨")
output.serve_kernel_port_as_window(FRONTEND_PORT, path="/", skip_warning=True)


/content
Mounted at /content/drive
📥 Clonando repositorio...
📦 Instalando dependencias...
🛠️ Cargando modelos...
✅ Modelo 'whisper' cargado
✅ Modelo 'community-1' cargado
Non-A100 GPU detected, using math or mem efficient attention if input tensor is on cuda
✅ Modelo 'bs_roformer' cargado
🏗️  Preparando OpenReel (deps)...

🛠️ Iniciando APP...


INFO:     Started server process [255]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



❌ El frontend no arrancó (o murió). Últimas líneas:
> @openreel/web@0.1.0 dev /content/v2v/openreel/apps/web
> vite "--" "--host" "0.0.0.0" "--port" "5173" "--strictPort"
VITE v5.4.21  ready in 292 ms
➜  Local:   http://localhost:5173/
➜  Network: use --host to expose
[baseline-browser-mapping] The data in this module is over two months old.  To ensure accurate Baseline data, please update: `npm i baseline-browser-mapping@latest -D`


RuntimeError: Vite no está escuchando en el puerto del frontend

In [2]:
output.serve_kernel_port_as_window(FRONTEND_PORT, path="/", skip_warning=True)

<IPython.core.display.Javascript object>